# PredictGuard — Phase 1, Stage 4: Leakage-Safe Data Splitting

**Project**: Explainable Predictive Maintenance System  
**Dataset**: Microsoft Azure Predictive Maintenance  
**Author**: PredictGuard Contributors  
**Stage**: 4 of 6 — Leakage-Safe Data Splitting

---

## Objective

Construct a **reproducible, machine-level leakage-safe data splitting pipeline** for predictive maintenance.  
All reusable logic lives in `src/split.py`.

### Why Machine-Level Splitting is Required

> Standard random row splitting (`train_test_split(X, y)`) causes catastrophic temporal data leakage in time-series predictive maintenance. Highly correlated adjacent telemetry timestamps from the *same physical machine* bleed across train and test sets, inflating test metrics artificially while failing when deployed to unseen equipment.

### Split Strategy
1. **Development Set (80 Machines)**: Used for model training and 5-fold Grouped Cross-Validation.
2. **Final Test Set (20 Machines)**: Held out completely untouched until final evaluation.
3. **Grouped Cross-Validation**: `StratifiedGroupKFold` grouped by `machineID` on Development set.

---
## 0. Environment Setup & Imports

In [ ]:
import logging
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Ensure src/ is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import split as sp

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("stage4_notebook")
logger.info("Stage 4 notebook started.")

In [ ]:
PROCESSED_DIR = project_root / "data" / "processed"
REPORTS_DIR = project_root / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Load Features Dataset

In [ ]:
df_features = pd.read_parquet(PROCESSED_DIR / "features.parquet")
print(f"Loaded feature dataset: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns across {df_features['machineID'].nunique()} machines.")

---
## 2. Machine-Level Development / Test Split

> **Machine-Level Stratification**: Machines are split using `MachineSplitter(test_ratio=0.20, random_seed=42)` stratified by machine failure status to ensure balanced failure representation in both sets.

In [ ]:
splitter = sp.MachineSplitter(test_ratio=0.20, random_seed=42, stratify_by_failure=True)
dev_df, test_df, dev_ids, test_ids = splitter.split(df_features)

print(f"Development Machines ({len(dev_ids)}): {dev_ids[:10]} ...")
print(f"Final Test Machines  ({len(test_ids)}):  {test_ids}")

---
## 3. Grouped Cross-Validation Folds

> **Grouped K-Fold**: Generates 5 cross-validation folds on the Development dataset grouped strictly by `machineID`.

In [ ]:
cv = sp.GroupedCrossValidator(n_folds=5, random_seed=42)
fold_assignments, fold_summary = cv.generate_folds(dev_df)
display(fold_summary)

---
## 4. Exhaustive Leakage Validation Check

In [ ]:
val_results = sp.LeakageValidator.validate_split(dev_df, test_df, fold_assignments)
print("\n✅ All Leakage Validation checks passed with zero errors!")

---
## 5. Compute Statistics & Save Manifest Outputs

In [ ]:
stats = sp.compute_split_statistics(dev_df, test_df)

# Save Machine ID CSVs
pd.DataFrame({"machineID": dev_ids}).to_csv(PROCESSED_DIR / "development_machine_ids.csv", index=False)
pd.DataFrame({"machineID": test_ids}).to_csv(PROCESSED_DIR / "test_machine_ids.csv", index=False)

# Save Parquets
dev_df.to_parquet(PROCESSED_DIR / "development.parquet", index=False, engine="pyarrow")
test_df.to_parquet(PROCESSED_DIR / "test.parquet", index=False, engine="pyarrow")

# Save Manifest & Report
sp.save_split_manifest({"test_ratio": 0.20, "random_seed": 42}, dev_ids, test_ids, PROCESSED_DIR / "split_manifest.json")
sp.generate_split_report(stats, fold_summary, REPORTS_DIR / "split_strategy_report.md")

print("✅ All datasets, JSON manifests, and markdown report saved successfully.")

---
## 6. Generate Split Visualisations

In [ ]:
sp.plot_split_summary_report(dev_df, test_df, fold_assignments, FIGURES_DIR)
print("✅ Unified Split Summary Report figure saved under reports/figures/split_summary_report.png")

---
## 7. Stage 4 Summary & Next Steps

| Deliverable | Status |
|---|---|
| Machine-Level Split (80 Dev / 20 Test) | ✅ Executed |
| 5-Fold Grouped Cross-Validation | ✅ Generated |
| Leakage Validation Check | ✅ 100% Leakage-Free |
| Parquets (`development.parquet`, `test.parquet`) | ✅ Saved to `data/processed/` |
| Split Manifest (`split_manifest.json`) | ✅ Saved to `data/processed/` |
| Markdown Report | ✅ Saved to `reports/split_strategy_report.md` |

### Ready for Phase 2 — Stage 5 (Modelling & Evaluation)